# Dialog2Flow — first contextual-turn-embeddings smoke experiment (Colab)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jumafernandez/doctorado-unsl/blob/main/contextual-turn-embeddings/notebooks/colab_d2f_smoke.ipynb)

A **small** end-to-end smoke run of the contextual turn encoder (`f2`) on
**precomputed Dialog2Flow embeddings**:

clone repo → install package → package smoke test → mount Drive → load D2F
metadata + precomputed embeddings → build a canonical per-turn DataFrame
(embeddings stay row-aligned) → keep the first 1000 **complete** dialogues →
train 1 epoch (bidirectional, masked-reconstruction only) → export contextual
embeddings + metadata + config + checkpoint → sanity checks.

**Scope / guardrails**
- **Precomputed embeddings only** — no Hugging Face / SentenceTransformers downloads.
- **No** ANN / FAISS / MSS@10 / LLM-judging / statistical tests (that is a later stage).
- Not the full dataset — a 1000-dialogue subset.

**Before running:** commit & push `contextual-turn-embeddings/` to
`github.com/jumafernandez/doctorado-unsl` (this notebook clones it). For a private
repo, clone with a personal access token.

## 1–3. Clone the repo, enter the package, install (editable)

In [ ]:
import os

REPO_URL = "https://github.com/jumafernandez/doctorado-unsl"
REPO_DIR = "/content/doctorado-unsl"
PACKAGE_SUBDIR = "contextual-turn-embeddings"

# Make sure the package has been committed & pushed to the repo first.
# For a PRIVATE repo, clone with a token, e.g.:
#   https://<TOKEN>@github.com/jumafernandez/doctorado-unsl
if not os.path.isdir(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    print("Repository already present at", REPO_DIR)

%cd {REPO_DIR}/{PACKAGE_SUBDIR}
!pwd

In [ ]:
# Install the package (editable) + core deps. Precomputed embeddings only,
# so transformers / sentence-transformers are NOT needed here.
!pip install -q -r requirements.txt
!pip install -q pytest
!pip install -q -e .
print("Install complete.")

## 4. Run the package smoke test (mocked embeddings, no downloads)

In [ ]:
!python scripts/smoke_test.py

## 5. Mount Google Drive

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

## 6. Configurable paths

In [ ]:
import os

# >>> Change this base directory if your files live elsewhere <<<
BASE_DIR = "/content/drive/MyDrive/Doctorado/Cursos/ANN/TF"

DIALOGS_PKL    = os.path.join(BASE_DIR, "dialogs-2.0.pkl")
IDS_NPY        = os.path.join(BASE_DIR, "ids.npy")
EMBEDDINGS_NPY = os.path.join(BASE_DIR, "embeddings_dialog2flow.npy")

# Outputs: contextual embeddings export + model checkpoint subdir.
OUTPUT_DIR = os.path.join(BASE_DIR, "resultados", "contextual_turn_embeddings", "d2f_smoke")
MODEL_DIR  = os.path.join(OUTPUT_DIR, "model")

# Smoke-experiment size: number of COMPLETE dialogues to keep.
N_DIALOGUES = 1000

os.makedirs(OUTPUT_DIR, exist_ok=True)
print("Inputs:")
for p in [DIALOGS_PKL, IDS_NPY, EMBEDDINGS_NPY]:
    print(("  OK   " if os.path.exists(p) else "  MISS "), p)
print("Output dir:", OUTPUT_DIR)

## 7. Load Dialog2Flow metadata + precomputed embeddings (with diagnostics)

In [ ]:
import pickle
import numpy as np
import pandas as pd

with open(DIALOGS_PKL, "rb") as fh:
    dialogs = pickle.load(fh)

ids = np.load(IDS_NPY, allow_pickle=True)
embeddings = np.asarray(np.load(EMBEDDINGS_NPY), dtype=np.float32)

print("dialogs   :", type(dialogs))
if isinstance(dialogs, pd.DataFrame):
    print("  columns :", list(dialogs.columns))
    display(dialogs.head(3))
elif isinstance(dialogs, dict):
    keys = list(dialogs.keys())
    print("  n keys  :", len(keys), "| sample key:", keys[0])
    print("  sample value type:", type(dialogs[keys[0]]))
elif isinstance(dialogs, (list, tuple)):
    print("  length  :", len(dialogs), "| sample item type:", type(dialogs[0]))
    print("  sample  :", dialogs[0])

ids_arr = np.asarray(ids)
print("ids       :", ids_arr.shape, ids_arr.dtype, "| sample:", ids_arr.ravel()[:4])
print("embeddings:", embeddings.shape, embeddings.dtype)

## 8. Build a canonical, row-aligned DataFrame

Canonical columns: `row_id, dialogue_id, turn_id, utterance` (+ `speaker` if
available). Row `i` of `embeddings_aligned` must correspond to row `i` of `df`.

> ⚠️ Dialog2Flow exports vary. Look at the diagnostics in the previous cell and
> adjust the candidate field names below if the structure differs.

In [ ]:
# Robust adapter:
#   * flatten the metadata into per-turn rows (DataFrame / dict / list);
#   * if ids.npy encodes (dialogue_id, turn_id), order rows by ids (a join) so
#     they line up with the embedding rows exactly;
#   * otherwise assume the flattened order already matches the embeddings
#     (positional alignment) and assert the lengths agree.

SPEAKER_KEYS   = ("speaker", "role", "spk", "agent")
UTTERANCE_KEYS = ("utterance", "text", "utt", "content", "message")
TURNS_KEYS     = ("turns", "log", "utterances", "dialogue", "dialog")
DID_KEYS       = ("dialogue_id", "dialog_id", "id", "conv_id", "conversation_id")


def _first_key(d, keys, default=None):
    for k in keys:
        if k in d:
            return d[k]
    return default


def _coerce_turn(turn):
    if isinstance(turn, dict):
        return _first_key(turn, SPEAKER_KEYS), str(_first_key(turn, UTTERANCE_KEYS, ""))
    if isinstance(turn, (list, tuple)) and len(turn) >= 2:
        return turn[0], str(turn[1])
    return None, str(turn)


def flatten_dialogs(dialogs):
    if isinstance(dialogs, pd.DataFrame):
        df = dialogs.copy()
        ren = {}
        for cand in DID_KEYS:
            if cand in df.columns:
                ren[cand] = "dialogue_id"; break
        for cand in ("turn_id", "turn_idx", "turn", "index"):
            if cand in df.columns:
                ren[cand] = "turn_id"; break
        for cand in UTTERANCE_KEYS:
            if cand in df.columns:
                ren[cand] = "utterance"; break
        for cand in SPEAKER_KEYS:
            if cand in df.columns:
                ren[cand] = "speaker"; break
        df = df.rename(columns=ren)
        if "turn_id" not in df.columns:
            df["turn_id"] = df.groupby("dialogue_id").cumcount()
        return df
    rows = []
    items = dialogs.items() if isinstance(dialogs, dict) else enumerate(dialogs)
    for did, dlg in items:
        if isinstance(dlg, dict):
            real_id = _first_key(dlg, DID_KEYS, did)
            turns = _first_key(dlg, TURNS_KEYS, [])
        else:
            real_id, turns = did, dlg
        for t_idx, turn in enumerate(turns):
            spk, utt = _coerce_turn(turn)
            rows.append({"dialogue_id": real_id, "turn_id": t_idx,
                         "speaker": spk, "utterance": utt})
    return pd.DataFrame(rows)


def parse_ids_to_keys(ids):
    a = np.asarray(ids)
    if a.ndim == 2 and a.shape[1] >= 2:
        return pd.DataFrame({"dialogue_id": a[:, 0], "turn_id": a[:, 1]})
    if a.ndim == 1 and a.dtype.kind in ("U", "S", "O"):
        sample = str(a[0])
        for sep in ("__", "::", "_", "-", ":", "/"):
            if sep in sample:
                parts = [str(x).rsplit(sep, 1) for x in a]
                if all(len(p) == 2 for p in parts):
                    try:
                        tids = [int(p[1]) for p in parts]
                    except ValueError:
                        continue
                    return pd.DataFrame({"dialogue_id": [p[0] for p in parts], "turn_id": tids})
    return None


meta_df = flatten_dialogs(dialogs)
print("flattened metadata rows:", len(meta_df))

ids_keys = parse_ids_to_keys(ids)
if ids_keys is not None and len(ids_keys) == len(embeddings):
    lut = meta_df.copy()
    lut["dialogue_id"] = lut["dialogue_id"].astype(str)
    lut["turn_id"] = lut["turn_id"].astype(int)
    ids_keys["dialogue_id"] = ids_keys["dialogue_id"].astype(str)
    ids_keys["turn_id"] = ids_keys["turn_id"].astype(int)
    df = ids_keys.merge(lut, on=["dialogue_id", "turn_id"], how="left")
    print("Aligned via ids.npy (join on dialogue_id + turn_id).")
else:
    assert len(meta_df) == len(embeddings), (
        f"metadata rows ({len(meta_df)}) != embeddings ({len(embeddings)}); "
        "cannot align positionally - adapt this cell to your data."
    )
    df = meta_df.reset_index(drop=True)
    print("Aligned positionally (metadata order assumed == embeddings order).")

# Finalize canonical columns.
if "utterance" not in df.columns:
    df["utterance"] = ""
df["utterance"] = df["utterance"].fillna("").astype(str)

# Drop an all-empty speaker column so speaker embeddings stay optional.
if "speaker" in df.columns and not df["speaker"].notna().any():
    df = df.drop(columns=["speaker"])

df.insert(0, "row_id", np.arange(len(df), dtype=np.int64))
embeddings_aligned = embeddings  # row i <-> df row i

keep = [c for c in ["row_id", "dialogue_id", "turn_id", "utterance", "speaker"] if c in df.columns]
df = df[keep]

assert len(df) == len(embeddings_aligned), "df and embeddings are not aligned"
print("canonical df:", df.shape, "| columns:", list(df.columns))
df.head()

## 9. Select a subset of complete dialogues (no random turns)

In [ ]:
# Keep the first N COMPLETE dialogues (all of their turns), never random turns.
ordered_dialogues = pd.unique(df["dialogue_id"])
selected = set(ordered_dialogues[:N_DIALOGUES])
mask = df["dialogue_id"].isin(selected).to_numpy()

df_subset = df[mask].reset_index(drop=True)
emb_subset = embeddings_aligned[mask]

print("selected dialogues :", df_subset["dialogue_id"].nunique())
print("rows (turns)       :", len(df_subset))
print("embeddings subset  :", emb_subset.shape)
df_subset.head()

## 10. Train `ContextualTurnModel` (bidirectional, masked-only, 1 epoch)

In [ ]:
from contextual_turn_embeddings import Config, ModelConfig, train

has_speaker = "speaker" in df_subset.columns
input_dim = emb_subset.shape[1]   # expected 768 for Dialog2Flow

config = Config()
config.model = ModelConfig(
    input_dim=input_dim,
    hidden_dim=input_dim,
    output_dim=input_dim,          # output dim == input dim
    num_layers=4,
    num_heads=8,
    dropout=0.1,
    max_turns=64,
    attention_mode="bidirectional",
    use_speaker_embeddings=has_speaker,
    num_speakers=4,
)
# Masked reconstruction only (next-turn is leaky in bidirectional mode).
config.losses.masked_reconstruction.enabled = True
config.losses.next_turn_prediction.enabled = False

config.training.epochs = 1
config.training.batch_size = 32
config.training.device = "auto"
config.training.output_dir = MODEL_DIR

config.data.max_turns = 64

model = train(config, df=df_subset, embeddings=emb_subset)

## 11. Export contextual embeddings to Drive

In [ ]:
from contextual_turn_embeddings import encode_dialogues, export, get_device

device = str(get_device(config.training.device))
matrix, metadata = encode_dialogues(
    model,
    df_subset,
    embeddings=emb_subset,
    data_config=config.data,
    device=device,
)
export(OUTPUT_DIR, matrix, metadata, config=config)

print("Exported to:", OUTPUT_DIR)
print("  - contextual_embeddings.npy", matrix.shape)
print("  - metadata.csv            ", metadata.shape)
print("  - config.json")
print("  - model checkpoint dir    :", MODEL_DIR)

## 12. Sanity checks

In [ ]:
import json
import numpy as np

# Alignment & integrity
assert matrix.shape[0] == len(metadata), "row count mismatch"
assert matrix.shape[1] == emb_subset.shape[1], "output dim != input dim"
assert not np.isnan(matrix).any(), "NaNs found in contextual embeddings"

n_dialogues = metadata["dialogue_id"].nunique()
n_turns = len(metadata)
avg_turns = n_turns / max(1, n_dialogues)

print("Sanity checks passed.")
print(f"dialogues            : {n_dialogues}")
print(f"turns                : {n_turns}")
print(f"avg turns / dialogue : {avg_turns:.2f}")
print(f"input / base dim     : {emb_subset.shape[1]}")
print(f"contextual dim       : {matrix.shape[1]}")
print(f"contextual matrix    : {matrix.shape}")

# Final training loss (read from the checkpoint's training log, if present)
log_path = os.path.join(MODEL_DIR, "training_log.jsonl")
if os.path.exists(log_path):
    with open(log_path) as fh:
        entries = [json.loads(line) for line in fh if line.strip()]
    if entries:
        print("final training step  :", entries[-1])

# Verify exported files exist
print("Exported files:")
for fn in ["contextual_embeddings.npy", "metadata.csv", "config.json"]:
    print(("  OK   " if os.path.exists(os.path.join(OUTPUT_DIR, fn)) else "  MISS "), fn)

metadata.head()